In [1]:
from model import Transformer

ModuleNotFoundError: No module named 'einops'

## Load up transformer

In [ ]:
seq_len = 256
model_dim = 64
heads = 8
model = Transformer(vocab_size=50258, dim=model_dim, depth=8, heads=heads, dim_head=model_dim//heads, mlp_dim=128, seq_len=seq_len)

In [ ]:
model_path = "Checkpoints/model_epoch_100.pt"
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

In [ ]:
# convert logits to top k tokens

def logits_to_tokens(logits, top_k = 5):
    # Get the top_k values and their corresponding indices from the logits
    values, indices = torck.topk(logits, top_k)

    # create mask for logits
    mask = torch.full(logits.shape, -float('inf')).to(device)
    mask.scatter_(1, indices, values) # set top k to original values

    return mask

In [ ]:
# Function to sample from the model autoregressively
def sample_from_model(model, tokenizer, device, max_length=50, temperature=0.1, top_k=5, additional_text):
    # Start with the BOS token
    input_ids = tokenizer.encode(tokenizer.bos_token, return_tensors='pt').to(device)
    
    input_ids = torch.cat([input_ids, tokenizer.encode(additional_input_text, return_tensors='pt').to(device)], dim=-1)
    
    output_ids = input_ids
    
    # Generate tokens
    with torch.no_grad():
        for _ in range(max_length):
            embeddings = model.embed(input_ids)
            outputs = model(embeddings)
            next_token_logits = outputs[:, -1, :] / temperature
            
            next_token_logits = logits_to_tokens(next_token_logits, top_k=5).to(device)
            
            next_token = torch.multinomial(torch.softmax(next_token_logits, dim=-1), num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            output_ids = torch.cat([output_ids, next_token], dim=-1)
            
            # trim input to sequence length
            input_ids = input_ids[:, -model.seq_len:]
            
            if next_token.item() == tokenizer.eos_token_id:
                break

    # Decode the generated tokens
    generated_text = tokenizer.decode(output_ids[0].cpu(), skip_special_tokens=True)
    return generated_text

In [ ]:
generated_text = sample_from_model(model, tokenizer, device, max_length=100, temperature=0.1, top_k=50)
print(len(generated_text))

words_per_line = 10

generated_text = " ".join(generated_text.split())

# Split the text into lines with a specified number of words per line
words = generated_text.split()
lines = [
    " ".join(words[i:i+words_per_line]) 
    for i in range(0, len(words), words_per_line)
]

# Join the lines with newlines
generated_text = "\n".join(lines)

print(generated_text)